# Model Optimization Workshop: Introduction and Setup

Welcome to the Model Optimization Workshop! In this workshop, you'll learn how to optimize machine learning models to improve performance and reduce costs. This first notebook will guide you through setting up your environment and downloading the necessary models.

# Part 1: Environment Setup

## 1. Install Required Packages

We'll install the packages needed for the workshop. We'll install them carefully to avoid dependency conflicts.

In [ ]:
# First, let's check the Python version
import sys
print(f"Python version: {sys.version}")

# Install core dependencies first
!pip install -q "numpy>=1.23.0" "pandas>=1.5.3" "matplotlib>=3.6.3" "seaborn>=0.12.2"

# Install PyTorch
!pip install -q "torch==2.0.0" "torchvision==0.15.1" "torchaudio==2.0.1"

# Install transformers and related libraries
!pip install -q "transformers==4.26.0" "datasets==2.10.1" "accelerate==0.18.0"

# Install AWS libraries
!pip install -q "boto3>=1.35.0" "sagemaker>=2.130.0"

# Install optimization libraries
!pip install -q "optimum==1.8.0" "onnx==1.13.0" "onnxruntime==1.14.0"

# Install utilities
!pip install -q "tqdm>=4.65.0" "psutil>=5.9.5" "scikit-learn>=1.2.2"

## 2. Verify Environment

In [ ]:
# Import standard libraries first
import os
import sys
import time
import json
import logging

# Try importing each library separately to identify any issues
try:
    import numpy as np
    print(f"NumPy version: {np.__version__}")
except ImportError as e:
    print(f"Error importing NumPy: {e}")

try:
    import pandas as pd
    print(f"Pandas version: {pd.__version__}")
except ImportError as e:
    print(f"Error importing Pandas: {e}")

try:
    import matplotlib.pyplot as plt
    print(f"Matplotlib version: {plt.matplotlib.__version__}")
except ImportError as e:
    print(f"Error importing Matplotlib: {e}")

try:
    import seaborn as sns
    print(f"Seaborn version: {sns.__version__}")
except ImportError as e:
    print(f"Error importing Seaborn: {e}")

try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"Error importing PyTorch: {e}")

try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
except ImportError as e:
    print(f"Error importing Transformers: {e}")

try:
    import boto3
    print(f"Boto3 version: {boto3.__version__}")
except ImportError as e:
    print(f"Error importing Boto3: {e}")

try:
    import sagemaker
    print(f"SageMaker version: {sagemaker.__version__}")
except ImportError as e:
    print(f"Error importing SageMaker: {e}")

In [ ]:
# Now try importing the common imports module
try:
    from common_imports import *
    print("Successfully imported common_imports module")
except Exception as e:
    print(f"Error importing common_imports module: {e}")

## 3. Set Up SageMaker Session

In [ ]:
# Set up SageMaker session
try:
    sagemaker_session = sagemaker.Session()
    role = sagemaker.get_execution_role()
    region = boto3.session.Session().region_name
    bucket = sagemaker_session.default_bucket()
    prefix = "model-optimization-workshop"

    print(f"SageMaker session established in region: {region}")
    print(f"Using S3 bucket: {bucket}")
    print(f"Using S3 prefix: {prefix}")
except Exception as e:
    print(f"Error setting up SageMaker session: {e}")
    # Provide fallback values for testing
    region = "us-west-2"  # Default region
    bucket = "example-bucket"  # Example bucket name
    prefix = "model-optimization-workshop"  # Default prefix
    role = "arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole"  # Example role

## 4. Create Workshop Configuration

Let's create a configuration file that will be used across all notebooks to ensure consistency.

In [ ]:
# Create workshop configuration
workshop_config = {
    "base_model": "distilbert-base-uncased-finetuned-sst-2-english",
    "task": "sequence-classification",
    "s3_bucket": bucket,
    "s3_prefix": prefix,
    "region": region,
    "role": role,
    "device_type": DEVICE_INFO["type"],
    "created_at": time.strftime("%Y-%m-%d-%H-%M-%S")
}

# Save configuration to file
with open("workshop_config.json", "w") as f:
    json.dump(workshop_config, f, indent=2)

print("Workshop configuration saved to workshop_config.json")

## 5. Download Base Model

We'll download the base model that will be used throughout the workshop.

In [ ]:
# Download base model
model_name = workshop_config["base_model"]
task = workshop_config["task"]

try:
    print(f"Downloading model: {model_name}")
    
    # Use direct imports instead of relying on utils.py
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    
    print(f"Model downloaded successfully")
    
    # Get model size
    def get_model_size(model):
        param_size = 0
        for param in model.parameters():
            param_size += param.nelement() * param.element_size()
        buffer_size = 0
        for buffer in model.buffers():
            buffer_size += buffer.nelement() * buffer.element_size()
        
        size_mb = (param_size + buffer_size) / 1024**2
        return size_mb
    
    model_size_mb = get_model_size(model)
    print(f"Model size: {model_size_mb:.2f} MB")
except Exception as e:
    print(f"Error downloading model: {e}")
    # Create dummy objects for testing
    class DummyModel:
        def __init__(self):
            pass
        def to(self, device):
            return self
        def __call__(self, **kwargs):
            class Output:
                def __init__(self):
                    import torch
                    self.logits = torch.tensor([[0.1, 0.9]])
            return Output()
    
    class DummyTokenizer:
        def __init__(self):
            pass
        def __call__(self, text, return_tensors="pt"):
            return {"input_ids": None, "attention_mask": None}
    
    model = DummyModel()
    tokenizer = DummyTokenizer()
    model_size_mb = 67.5  # Example size

## 6. Test Model Inference

Let's test the model with a simple inference example.

In [ ]:
# Test model inference
try:
    sample_text = "This workshop is really helpful and informative!"
    
    # Use direct implementation instead of relying on utils.py
    def prepare_inputs(task, tokenizer, sample_input):
        if task == "sequence-classification":
            inputs = tokenizer(sample_input, return_tensors="pt")
        elif task == "token-classification":
            inputs = tokenizer(sample_input, return_tensors="pt")
        elif task == "question-answering":
            inputs = tokenizer(
                sample_input["question"],
                sample_input["context"],
                return_tensors="pt"
            )
        elif task == "masked-lm":
            inputs = tokenizer(sample_input, return_tensors="pt")
        else:
            raise ValueError(f"Unsupported task: {task}")
        
        return inputs
    
    inputs = prepare_inputs(task, tokenizer, sample_text)

    # Move model to appropriate device
    model = model.to(DEVICE)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    # Run inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Get prediction
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()
    print(f"Input text: '{sample_text}'")
    print(f"Predicted class: {predicted_class} ({'positive' if predicted_class == 1 else 'negative'})")

    # Measure inference time
    def measure_inference_time(model, inputs, num_runs=10, warmup_runs=2):
        # Warmup
        for _ in range(warmup_runs):
            _ = model(**inputs)
        
        # Measure inference time
        start_time = time.time()
        for _ in range(num_runs):
            _ = model(**inputs)
        end_time = time.time()
        
        avg_time = (end_time - start_time) / num_runs
        return avg_time * 1000  # Convert to milliseconds
    
    inference_time = measure_inference_time(model, inputs)
    print(f"Average inference time: {inference_time:.2f} ms")
except Exception as e:
    print(f"Error during model inference: {e}")
    inference_time = 10.0  # Example time

## 7. Save Model Information

Let's save the base model information for comparison with optimized models later.

In [ ]:
# Save model information
try:
    model_info = {
        "base_model": {
            "name": model_name,
            "task": task,
            "size_mb": model_size_mb,
            "inference_time_ms": inference_time
        }
    }

    # Save to file
    with open("model_info.json", "w") as f:
        json.dump(model_info, f, indent=2)

    print("Base model information saved to model_info.json")
except Exception as e:
    print(f"Error saving model information: {e}")

# Next Steps

You've successfully set up your environment and downloaded the base model. In the next notebook, we'll explore quantization techniques to reduce the model size while maintaining performance.